# AI Investment Risk Radar: End-to-End Execution Pipeline

This notebook demonstrates the end-to-end AI Investment Risk Radar quantitative pipeline using modular components from `ai_risk_radar`. It reproduces data loading/acquisition, causal feature engineering, purged model training and Platt calibration, validation strategy selection, vectorized baseline approximation reconciliation, and latest market risk inference with an estimated trade schedule.

In [1]:
import os
import sys
from pathlib import Path

import exchange_calendars as xcals
import pandas as pd

# Add project root to sys.path
cur_dir = Path.cwd()
if (cur_dir / "config.json").is_file():
    root_dir = cur_dir
elif (cur_dir.parent / "config.json").is_file():
    root_dir = cur_dir.parent
else:
    root_dir = cur_dir
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from ai_risk_radar.analysis import compute_regime_analysis
from ai_risk_radar.config import load_settings
from ai_risk_radar.data import (
    create_offline_synthetic_fixture,
    fetch_prices,
    load_snapshot,
    save_snapshot,
)
from ai_risk_radar.features import build_features
from ai_risk_radar.models import fit_models, predict_risk
from ai_risk_radar.pipeline import format_threshold_key, parse_threshold_key
from ai_risk_radar.simulation import (
    benchmark_targets,
    reconcile_vectorized_and_sequential,
    select_strategy,
    simulate,
    weekly_targets,
)
from ai_risk_radar.splits import make_splits

settings = load_settings(root_dir / "config.json")
print(f"Config loaded from {root_dir / 'config.json'}")
print(f"Target definition: {settings.target_horizon}-session return <= {settings.target_threshold:.1%}")
print(f"Universe tickers: {settings.tickers}")


Config loaded from .\config.json
Target definition: 20-session return <= -5.0%
Universe tickers: ('SOXX', 'XLK', 'SPY', 'SHY')


## 1. Clean First-Use Data Acquisition & Snapshot Verification

Load the verified evaluation snapshot from `data/raw/snapshot`. On a clean checkout where data has not yet been cached, the cell initiates explicit acquisition via `fetch_prices`. If running offline or without network access, it engages a deterministic offline synthetic fixture. Data provenance is tracked explicitly and never silently conflated.

In [2]:
mode = os.environ.get("AI_RISK_RADAR_NOTEBOOK_MODE")
allowed_modes = {"fixture", "cache", "live", None}
if mode not in allowed_modes:
    raise ValueError(f"Unsupported notebook mode: {mode}")

if mode == "live":
    print("[LIVE DOWNLOAD] Acquiring latest market data via yfinance...")
    bundle = fetch_prices(settings)
    data_provenance = "live_download"
elif mode == "fixture":
    fixture_dir = root_dir / settings.raw_dir / "notebook_fixture"
    if not fixture_dir.is_dir() or not (fixture_dir / "manifest.json").is_file():
        bundle = create_offline_synthetic_fixture(settings, random_seed=settings.random_seed)
        save_snapshot(bundle, fixture_dir)
    else:
        bundle = load_snapshot(fixture_dir)
    data_provenance = "offline_synthetic_fixture"
    print("[SYNTHETIC DEMONSTRATION] Engaging deterministic offline fixture.")
    print("[NOT MARKET EVIDENCE] Synthetic data is for pipeline demonstration only.")
else:
    snapshot_dir = root_dir / settings.raw_dir / "snapshot"
    if snapshot_dir.is_dir() and (snapshot_dir / "manifest.json").is_file():
        print(f"[CACHED EVALUATION SNAPSHOT] Loading existing snapshot from: {snapshot_dir}")
        bundle = load_snapshot(snapshot_dir)
        data_provenance = "cached_snapshot"
    else:
        raise FileNotFoundError(
            f"No cached notebook data found at {snapshot_dir}. "
            "Please run acquisition or set AI_RISK_RADAR_NOTEBOOK_MODE='fixture'."
        )

print(f"\nActive Data Provenance: {data_provenance}")
print(f"Trading Sessions:       {len(bundle.sessions)} ({bundle.sessions[0].date()} to {bundle.sessions[-1].date()})")
print(f"Universe Tickers:       {bundle.manifest['tickers']}")
print(f"Manifest Source:        {bundle.manifest.get('source')}")


[SYNTHETIC DEMONSTRATION] Engaging deterministic offline fixture.
[NOT MARKET EVIDENCE] Synthetic data is for pipeline demonstration only.

Active Data Provenance: offline_synthetic_fixture
Trading Sessions:       2734 (2016-01-04 to 2026-11-16)
Universe Tickers:       ['SOXX', 'XLK', 'SPY', 'SHY']
Manifest Source:        deterministic_offline_fixture


## 2. Causal Feature Generation & Purged Temporal Splits

Construct 43 price-derived technical features (returns, realized volatilities, drawdowns, relative strength, correlations) strictly from backward-looking prices ($t-1$). Purge the target label horizon (20 trading days) at partition boundaries to ensure zero lookahead leak across train, calibration, validation, and test splits.

In [3]:
features = build_features(bundle, settings)
splits = make_splits(features, settings)
print(f"Generated {len(features.feature_columns)} causal technical features strictly from backward-looking prices (t-1).")
print(f"Train sessions:          {len(splits.train):>5d} ({splits.train[0].date()} to {splits.train[-1].date()})")
print(f"Calibration sessions:    {len(splits.calibration):>5d} ({splits.calibration[0].date()} to {splits.calibration[-1].date()})")
print(f"Validation sessions:     {len(splits.validation):>5d} ({splits.validation[0].date()} to {splits.validation[-1].date()})")
print(f"Test sessions (holdout): {len(splits.test):>5d} ({splits.test[0].date()} to {splits.test[-1].date()})")


Generated 43 causal technical features strictly from backward-looking prices (t-1).
Train sessions:           1398 (2016-04-05 to 2021-10-20)
Calibration sessions:      232 (2021-11-18 to 2022-10-20)
Validation sessions:       481 (2022-11-18 to 2024-10-18)
Test sessions (holdout):   500 (2024-11-18 to 2026-11-16)


## 3. Train & Calibrate Random Forest Downside Predictor

Fit candidate models on the purged training set with expanding walk-forward cross-validation. Apply Platt scaling on the calibration partition to ensure forecast values are well-calibrated probabilities rather than uncalibrated tree vote fractions.

In [4]:
models = fit_models(features, splits, settings)
rf_bundle = models["random_forest"]
dt_bundle = models["decision_tree"]
cb_bundle = models["constant_baseline"]
hgb_bundle = models["hist_gradient_boosting"]

print("=" * 72)
print("PURGED TEMPORAL MODEL COMPARATOR EVALUATION (Training Split)")
print("=" * 72)
print(f"{'Model Name':<28s} | {'CV Brier Score':>16s} | {'Status':>14s} | {'Selected Params'}")
print("-" * 72)

for name, bnd in [
    ("Constant Baseline", cb_bundle),
    ("Decision Tree", dt_bundle),
    ("HistGradientBoosting", hgb_bundle),
    ("Random Forest (Production)", rf_bundle),
]:
    cv_b = bnd.metadata.get("cv_brier_score")
    cv_str = f"{cv_b:.4f}" if cv_b is not None else "N/A"
    status = bnd.metadata.get("probability_status", "uncalibrated")
    depth = bnd.metadata.get("selected_depth") or bnd.metadata.get("selected_max_depth")
    leaf = bnd.metadata.get("selected_min_leaf") or bnd.metadata.get("selected_min_samples_leaf")
    params = f"depth={depth}" + (f", leaf={leaf}" if leaf else "") if depth else "N/A"
    print(f"{name:<28s} | {cv_str:>16s} | {status:>14s} | {params}")

print("=" * 72)
print(f"Trained Random Forest:          {rf_bundle.metadata['model_name']}")
print(f"Probability Calibration Status: {rf_bundle.metadata.get('probability_status')}")
print(f"Calibration Reason / Support:   {rf_bundle.metadata.get('reason')}")


PURGED TEMPORAL MODEL COMPARATOR EVALUATION (Training Split)
Model Name                   |   CV Brier Score |         Status | Selected Params
------------------------------------------------------------------------
Constant Baseline            |              N/A |   uncalibrated | N/A
Decision Tree                |           0.0979 |   uncalibrated | depth=2
HistGradientBoosting         |           0.0640 |   uncalibrated | depth=3
Random Forest (Production)   |           0.0497 |     calibrated | depth=3, leaf=50
Trained Random Forest:          random_forest
Probability Calibration Status: calibrated
Calibration Reason / Support:   Calibration support adequate


## 4. Strategy Selection & Delayed Simulation on Validation Set

Simulate candidate threshold allocation policies on the validation partition using institutional delayed execution: weekly rebalance signals evaluated at session close execute at the subsequent trading session close with a 10 bps turnover transaction cost. Select the optimal policy objectively via `select_strategy` (minimizing maximum drawdown subject to achieving at least the baseline CAGR, tie-broken by higher CAGR and lower turnover).

In [5]:
df = features.frame
val_prices = bundle.adjusted_close.loc[splits.validation]
val_frame = df.loc[splits.validation]

# Generate robust risk predictions on validation partition via predict_risk
val_probs = predict_risk(rf_bundle, val_frame)

# Evaluate candidate threshold strategies preserving exact threshold keys
candidates = {}
threshold_by_key = {}
for th in settings.candidate_thresholds:
    key = format_threshold_key(th)
    threshold_by_key[key] = float(th)
    t_df = weekly_targets(val_probs, val_prices, settings, threshold=th)
    candidates[key] = simulate(val_prices, t_df, settings, cost_rate=settings.cost_rate)

# Evaluate benchmark baseline
benchmarks = benchmark_targets(bundle, settings, sessions=splits.validation)
baseline_res = simulate(val_prices, benchmarks["fixed_defensive_weekly"], settings, cost_rate=settings.cost_rate)

# Select optimal strategy via canonical rule with calibration gating
candidate_key = select_strategy(candidates, baseline_res)
if candidate_key is not None and rf_bundle.metadata.get("probability_status") == "calibrated":
    selected_key = candidate_key
    selected_res = candidates[selected_key]
    selected_threshold = threshold_by_key.get(selected_key, parse_threshold_key(selected_key))
    threshold_str = f"{selected_threshold:.4f}".rstrip("0").rstrip(".")
else:
    selected_key = "fixed_defensive_weekly"
    selected_res = baseline_res
    selected_threshold = None
    threshold_str = "None (Defensive Baseline Fallback)"
metrics = selected_res.metrics

print(f"Selected Strategy Policy: {selected_key} (Threshold: {threshold_str})")
print(f"  Total Return:        {metrics['total_return']:+.2%}")
print(f"  CAGR:                {metrics['cagr']:+.2%}")
print(f"  Max Drawdown:        {metrics['max_drawdown']:.2%}")
print(f"  Sharpe Ratio (rf=0): {metrics['sharpe']:.2f}")
print(f"  Turnover:            {metrics['turnover']:.2f}x")
print(f"  Defensive Occupancy: {metrics['defensive_occupancy']:.1%}")
print(f"  Total Costs:         ${metrics['total_cost']:.2f}")


Selected Strategy Policy: fixed_defensive_weekly (Threshold: None (Defensive Baseline Fallback))
  Total Return:        -4.78%
  CAGR:                -2.54%
  Max Drawdown:        -8.87%
  Sharpe Ratio (rf=0): -0.45
  Turnover:            1.06x
  Defensive Occupancy: 100.0%
  Total Costs:         $20.07


## 5. Reconcile Vectorized Matrix Baseline Against Sequential Execution Engine

Compare the lecture-style vectorized baseline against the exact daily-drifted sequential simulation engine under matched delayed execution and transaction cost conventions.

**Methodological Note on Approximation vs. Drift:**
- **Vectorized Linear Approximation:** Assumes target weights are static intra-week and computes first-order turnover friction: $r_t = \sum w_{t-1} R_t - c \sum |\Delta w_t|$.
- **Sequential Execution Engine:** Tracks exact physical share holdings drifting daily with individual asset prices, and solves the self-financing cash conservation equation via Brent's method ($V^{\text{post}} + \text{Cost} = V^{\text{pre}}$).
- **Interpreting Reconciliation:** Numerical comparison (~30–50 bps over multi-year backtests, e.g. ~35 bps on synthetic fixtures, 43.4 bps on validation data) serves as an empirical comparison and smoke check confirming that the linear approximation tracks the sequential engine reasonably well under similar conventions. Numerical proximity does not prove equivalent delayed execution or transaction cost mechanics, as the vectorized model fundamentally ignores intra-week holding drift and nonlinear cash constraints. Over different datasets or market regimes, the divergence varies naturally with price dispersion and rebalance frequency.
- **Zero-Drift Constant-Price Case:** Under constant prices (or identical returns across all assets with zero transaction costs), intra-week price drift is zero. This serves as a mathematical unit check where both models produce identical returns, but does not validate non-zero transaction cost solving or execution timing under real price movements.

In [6]:
if selected_threshold is not None:
    selected_targets = weekly_targets(val_probs, val_prices, settings, threshold=selected_threshold)
else:
    selected_targets = benchmarks["fixed_defensive_weekly"]
reconciliation = reconcile_vectorized_and_sequential(
    val_prices,
    selected_targets,
    settings,
    cost_rate=settings.cost_rate,
)

print("=" * 78)
print("Reconciliation Summary (Validation Partition):")
print(f"{'Metric':<22s} | {'Sequential (Exact)':>18s} | {'Vectorized (Approx)':>19s} | {'Difference':>12s}")
print("-" * 78)
for metric, row in reconciliation["comparison_table"].items():
    s_str = f"{row['sequential']:.4f}" if isinstance(row['sequential'], float) else str(row['sequential'])
    v_str = f"{row['vectorized']:.4f}" if isinstance(row['vectorized'], float) else str(row['vectorized'])
    diff_str = f"{row['absolute_diff']:+.4f}" if row['absolute_diff'] is not None else "N/A"
    print(f"{metric:<22s} | {s_str:>18s} | {v_str:>19s} | {diff_str:>12s}")

print("\nMethodological Takeaway:")
print(reconciliation["explanation"])

# Substantive Error and Regime Analysis
print("\n" + "=" * 78)
print(f"DEEP REGIME AND SUBGROUP ERROR ANALYSIS (Data Provenance: {data_provenance})")
if data_provenance == "offline_synthetic_fixture":
    print("[PROVENANCE NOTICE] Demonstrating causal regime analysis on offline synthetic fixture.")
    print("                    Outputs are for reproducible demonstration, NOT historical market evidence.")
else:
    print("[PROVENANCE NOTICE] Empirical regime analysis evaluated on historical market data.")
print("=" * 78)

regime_analysis = compute_regime_analysis(rf_bundle, features, splits)

print("\n1. Partition Brier Scores & Prevalence:")
print(f"{'Partition':<16s} | {'Sessions':>10s} | {'Prevalence':>12s} | {'Brier Score':>12s} | {'Mean Pred Prob':>16s}")
print("-" * 74)
for part, p_data in regime_analysis["partitions"].items():
    brier_val = f"{p_data['brier_score']:.4f}" if p_data['brier_score'] is not None else "N/A"
    print(f"{part.title():<16s} | {p_data['sessions']:>10d} | {p_data['prevalence']:>12.1%} | {brier_val:>12s} | {p_data['mean_predicted_probability']:>16.1%}")

print("\n2. Annual Breakdown (Validation Partition):")
print(f"{'Year':<8s} | {'Sessions':>10s} | {'Events':>8s} | {'Prevalence':>12s} | {'Brier Score':>12s} | {'Mean Pred Prob':>16s}")
print("-" * 74)
for year, y_data in regime_analysis["annual_breakdown"].items():
    print(f"{year:<8d} | {y_data['session_count']:>10d} | {y_data['event_count']:>8d} | {y_data['prevalence']:>12.1%} | {y_data['brier_score']:>12.4f} | {y_data['mean_pred_prob']:>16.1%}")

print("\n3. Macro Trend Regimes (S&P 500 63-Session Trend):")
print(f"{'Regime':<16s} | {'Sessions':>10s} | {'Events':>8s} | {'Prevalence':>12s} | {'Brier Score':>12s} | {'Mean Pred Prob':>16s}")
print("-" * 74)
for regime, r_data in regime_analysis["trend_regimes"].items():
    print(f"{regime.replace('_', ' ').title():<16s} | {r_data['session_count']:>10d} | {r_data['event_count']:>8d} | {r_data['prevalence']:>12.1%} | {r_data['brier_score']:>12.4f} | {r_data['mean_pred_prob']:>16.1%}")

print("\n4. Volatility Regimes (20-Session Realized Volatility):")
print(f"{'Regime':<16s} | {'Sessions':>10s} | {'Events':>8s} | {'Prevalence':>12s} | {'Brier Score':>12s} | {'Mean Pred Prob':>16s}")
print("-" * 74)
for regime, v_data in regime_analysis["volatility_regimes"].items():
    print(f"{regime.replace('_', ' ').title():<16s} | {v_data['session_count']:>10d} | {v_data['event_count']:>8d} | {v_data['prevalence']:>12.1%} | {v_data['brier_score']:>12.4f} | {v_data['mean_pred_prob']:>16.1%}")
print("=" * 78)

Reconciliation Summary (Validation Partition):
Metric                 | Sequential (Exact) | Vectorized (Approx) |   Difference
------------------------------------------------------------------------------
total_return           |            -0.0478 |             -0.0462 |      +0.0015
cagr                   |            -0.0254 |             -0.0245 |      +0.0008
max_drawdown           |            -0.0887 |             -0.0879 |      +0.0007
annualized_volatility  |             0.0538 |              0.0538 |      +0.0000
sharpe                 |            -0.4498 |             -0.4338 |      +0.0160
turnover               |             1.0598 |              0.5250 |      +0.5348
defensive_occupancy    |             1.0000 |              1.0000 |      +0.0000
total_cost             |            20.0729 |              9.9900 |     +10.0829

Methodological Takeaway:
The vectorized backtest is a linear matrix approximation: it assumes static target weights intra-week and subtracts a f

## 6. Latest Market Risk Inference & Estimated Trade Schedule

Perform inference on the latest completed market session using the selected validation policy ($P(\text{downside}) \ge \text{threshold}$). Output the estimated trade schedule based on reference capital, latest available close prices, and reference baseline holdings. Note that execution occurs at the next completed session close ($t+1$), so share quantities, trade values, and transaction costs are forward-looking estimates that settle at next market close.

In [7]:
import math

prob_status = rf_bundle.metadata.get('probability_status', 'uncalibrated')
prob_reason = rf_bundle.metadata.get('reason') or 'Model probability unavailable'

latest_session = df.index[-1]
latest_frame = df.loc[[latest_session]]
latest_p_series = predict_risk(rf_bundle, latest_frame)
raw_cal_p = (
    float(latest_p_series.iloc[0])
    if len(latest_p_series) > 0 and not pd.isna(latest_p_series.iloc[0])
    else None
)

is_prob_unavailable = (
    prob_status == 'unavailable'
    or raw_cal_p is None
    or (isinstance(raw_cal_p, float) and (math.isnan(raw_cal_p) or math.isinf(raw_cal_p)))
)

if is_prob_unavailable:
    latest_cal_p = None
    prob_display = f'Unavailable ({prob_reason})'
    risk_level = 'Unavailable'
    is_defensive = True
    threshold_display = 'None (Probability Unavailable Fallback)'
    target_weights = settings.weights_fixed_baseline
    active_stance = 'Defensive Baseline Fallback'
else:
    latest_cal_p = raw_cal_p
    prob_display = f'{latest_cal_p:.1%}'
    risk_level = 'High' if latest_cal_p >= 0.35 else ('Medium' if latest_cal_p >= 0.20 else 'Low')
    if selected_threshold is not None and prob_status == 'calibrated':
        is_defensive = latest_cal_p >= selected_threshold
        threshold_display = f'{selected_threshold:.4f}'.rstrip('0').rstrip('.')
        target_weights = settings.weights_defensive if is_defensive else settings.weights_growth
        active_stance = 'Defensive' if is_defensive else 'Growth'
    else:
        is_defensive = True
        threshold_display = 'None (Defensive Baseline Fallback)'
        target_weights = settings.weights_fixed_baseline
        active_stance = 'Defensive Baseline Fallback'

# Derive weekly schedule on XNYS exchange calendar
cal = xcals.get_calendar('XNYS')
week_start = pd.Timestamp(latest_session.date() - pd.Timedelta(days=latest_session.weekday()))
week_end = week_start + pd.Timedelta(days=6)
week_sessions = cal.sessions_in_range(week_start, week_end)
final_week_session = week_sessions[-1]
is_week_final = (latest_session == final_week_session)
next_weekly_exec = cal.next_session(final_week_session)

final_str = final_week_session.strftime('%Y-%m-%d')
exec_str = next_weekly_exec.strftime('%Y-%m-%d')

if is_week_final:
    signal_session_display = latest_session.strftime('%Y-%m-%d')
    scheduled_exec_display = f'{exec_str} (at Market Close)'
    rebalance_allowed = True
    exec_protocol_note = f'Market close on {exec_str} via limit-on-close / market-on-close order'
else:
    signal_session_display = (
        f"{latest_session.strftime('%Y-%m-%d')} "
        f'(Mid-week observation; weekly signal evaluates at week close {final_str})'
    )
    scheduled_exec_display = f'{exec_str} (at Market Close, following {final_str} close)'
    rebalance_allowed = False
    exec_protocol_note = (
        f'No immediate rebalance; weekly policy maintains current stance until market close on {exec_str} '
        f'following week close on {final_str}'
    )

print('=' * 68)
print('AI INVESTMENT RISK RADAR: LATEST MARKET INFERENCE')
print('=' * 68)
print(f'Signal Session (Market Close):      {signal_session_display}')
print(f'Downside Risk Probability (20-day): {prob_display}')
print(f'Risk Classification Level:          {risk_level}')
print(f'Selected Validation Threshold:      {threshold_display}')
print(f'Active Portfolio Policy Stance:     {active_stance}')
print(f'Scheduled Execution Session:        {scheduled_exec_display}')
print('=' * 68)

# Build estimated trade schedule for reference capital
portfolio_equity = float(settings.initial_capital)
latest_prices = bundle.adjusted_close.loc[latest_session]

schedule_records = []
total_dollar_turnover = 0.0

for i, ticker in enumerate(settings.tickers):
    tgt_weight = target_weights[i]
    tgt_dollars = portfolio_equity * tgt_weight
    price = float(latest_prices[ticker])
    tgt_shares = tgt_dollars / price

    # Compare against reference baseline stance for rebalance delta demonstration
    curr_weight = settings.weights_fixed_baseline[i]
    curr_dollars = portfolio_equity * curr_weight

    if rebalance_allowed:
        delta_dollars = tgt_dollars - curr_dollars
        delta_shares = delta_dollars / price
        total_dollar_turnover += abs(delta_dollars)
        action = 'BUY' if delta_dollars > 1.0 else ('SELL' if delta_dollars < -1.0 else 'HOLD')
    else:
        delta_dollars = 0.0
        delta_shares = 0.0
        action = 'HOLD'

    schedule_records.append({
        'Ticker': ticker,
        'Target Weight': f'{tgt_weight:.1%}',
        'Target Value ($)': f'${tgt_dollars:,.2f}',
        'Close Price ($)': f'${price:,.2f}',
        'Estimated Shares': f'{tgt_shares:,.3f}',
        'Rebalance Action': action,
        'Trade Value ($)': f'{delta_dollars:+,.2f}',
        'Trade Shares': f'{delta_shares:+,.3f}',
    })

schedule_df = pd.DataFrame(schedule_records)
estimated_tx_cost = total_dollar_turnover * settings.cost_rate
cost_bps = round(settings.cost_rate * 10000)

header_note = 'Execution at Next Weekly Close' if rebalance_allowed else 'No Mid-Week Rebalance'
print(f'\nESTIMATED LATEST TRADE SCHEDULE (Reference Portfolio Equity: ${portfolio_equity:,.2f}, {header_note}):')
print(schedule_df.to_string(index=False))
print('-' * 68)
print(f'Total Gross Trade Volume:           ${total_dollar_turnover:,.2f}')
print(f'Estimated Rebalance Cost ({cost_bps} bps):  ${estimated_tx_cost:,.2f}')
print(f'Execution Protocol:                 {exec_protocol_note}')
print('Note: Target shares, trade values, and transaction fees are forward-looking estimates based on session t close prices and reference baseline holdings. Actual trade execution and cash settlement occur at weekly policy execution sessions.')
print('=' * 68)


AI INVESTMENT RISK RADAR: LATEST MARKET INFERENCE
Signal Session (Market Close):      2026-11-16 (Mid-week observation; weekly signal evaluates at week close 2026-11-20)
Downside Risk Probability (20-day): 10.2%
Risk Classification Level:          Low
Selected Validation Threshold:      None (Defensive Baseline Fallback)
Active Portfolio Policy Stance:     Defensive Baseline Fallback
Scheduled Execution Session:        2026-11-23 (at Market Close, following 2026-11-20 close)

ESTIMATED LATEST TRADE SCHEDULE (Reference Portfolio Equity: $10,000.00, No Mid-Week Rebalance):
Ticker Target Weight Target Value ($) Close Price ($) Estimated Shares Rebalance Action Trade Value ($) Trade Shares
  SOXX         20.0%        $2,000.00         $351.73            5.686             HOLD           +0.00       +0.000
   XLK         20.0%        $2,000.00          $38.14           52.434             HOLD           +0.00       +0.000
   SPY         20.0%        $2,000.00          $76.46           26.157 